# 000 — Paper 2 figures

Every figure for Paper 2, one section each. Each section says **what the figure is**,
**what it reads**, **what it writes**, and then has a single cell that rebuilds it by
calling the script that owns it.

**The figures are built by scripts, not by this notebook.** The notebook is the
explanation and the driver. `00_Paper2_Figures.py` holds the actual drawing code, so a
figure can be rebuilt on the server without a kernel, and so re-running a cell here can
never produce something different from re-running the script there.

**Before any of this:** the clustering runs have to exist. That means
`rebuild_concat_cache.py --apply`, then `240` / `241` / `242`, then cell 7 of each for
the held-out variance, then `249` to merge the sweeps. The preflight cell below checks
what it can and tells you what is missing rather than failing halfway through a render.

| | |
|---|---|
| scripts | `00_Paper2_Figures.py` (FIG 1), `00_paper2_figures2_2.py` (FIG 2) |
| figures | FIG 1a / 1b / 1c / 1d, FIG 2 |
| output | `outputs/clustering/paper_figures/` |
| per figure | the PNG, a `_caption.txt`, and a `_patients.csv` |
| renders | pyvista / VTK offscreen, fsaverage pial surfaces |

## Setup and preflight

Nothing here draws anything. It resolves the paths, then checks the inputs every figure
depends on and prints what is missing.

In [ ]:
import os, sys, json, subprocess, time
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'functions'))
import lf_runs as LR

SCRIPT  = ROOT / '00_Paper2_Figures.py'
FIGDIR  = ROOT / 'outputs' / 'clustering' / 'paper_figures'
PEAKS   = ROOT / 'outputs' / 'clustering' / 'bsf_comparison' / 'heldout_peaks_cnmf.csv'
MESHES  = ROOT / 'outputs' / '250_recon' / 'fsaverage' / 'meshes'
COORDS  = (ROOT / 'outputs' / '250_recon' / 'fsaverage' / 'coords'
           / 'ALL_PATIENTS_contacts_fsaverage.csv')
FEATURE_SETS = ['concat_hg', 'concat_rawds', 'concat_bands5', 'concat_bands5z']


def sh(cmd):
    """Run a project script the way 240/242 do - same interpreter, same cwd, utf-8."""
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    r = subprocess.run([sys.executable, *[str(c) for c in cmd]], cwd=str(ROOT),
                       env={**os.environ, 'PYTHONIOENCODING': 'utf-8'})
    assert r.returncode == 0, f'failed: {cmd}'


# ---- preflight ---------------------------------------------------------------
problems = []
if not SCRIPT.exists():
    problems.append(f'missing {SCRIPT.name}')
if not PEAKS.exists():
    problems.append(f'missing {PEAKS.name} - run 249 to merge the held-out sweeps')
for f in (COORDS, MESHES / 'fsaverage_lh.gii', MESHES / 'fsaverage_rh.gii'):
    if not f.exists():
        problems.append(f'missing {f.relative_to(ROOT)}')

if PEAKS.exists():
    peak = {r.feature_set: int(r.k_peak) for r in pd.read_csv(PEAKS).itertuples()}
    print(f"{'feature set':<16} {'peak K':>6}  {'cnmf run':<18} loadings at peak K")
    for fs in FEATURE_SETS:
        k = peak.get(fs)
        if k is None:
            problems.append(f'{fs}: no held-out peak'); print(f'{fs:<16} {"--":>6}  MISSING')
            continue
        try:
            rd = LR.newest_run('cnmf', fs)
        except Exception:
            rd = None
        if rd is None:
            problems.append(f'{fs}: no cnmf run - run 242'); print(f'{fs:<16} {k:>6}  MISSING')
            continue
        g = rd / 'loadings_by_k' / f'G_k{k:02d}.npy'
        if not g.exists():
            problems.append(f'{fs}: 242 did not sweep K={k}')
        print(f'{fs:<16} {k:>6}  {rd.name:<18} {"yes" if g.exists() else "NO"}')

print()
if problems:
    print('NOT READY:'); [print('  -', p) for p in problems]
else:
    print('ready - every input FIG 1 needs is on disk')

---

# FIG 0 — the task and the cohort

Built by **`00_paper2_figure0_coverage.py`**, which writes **two** figures. Everything
downstream is computed on this electrode set, so it comes first.

## The paper's FIG 0 (`FIG0_cohort.png`)

### A — the task, through four electrodes

One auditory, one visual, one motor, one preparatory — **chosen by hand as
illustrations**, named in `EXEMPLARS` at the top of the script with the reading Lora
gave each. For each: where it sits on its own hemisphere (alone, large), and its three
conditions concatenated under the trial strip, on the warped axis every later figure
uses. The caption records each one's coordinates, gate margins, and whether it is in the
analysed cohort.

### B — every electrode the gate saw

Left, from above, right: the ones the gate kept in green, the ones it rejected in grey.

## The supplement, FIG S0 (`FIG0_cohort_supplement.png`)

**A** electrodes the gate saw (hatched) and kept (solid), per patient, in FIG 1's patient
colours · **B** the same on the brain, kept in the patient's colour, rejected grey ·
**C** of the kept electrodes, those with a LanA value · **D** the same on the brain,
orange vs grey · **E / F** the electrode that just passed the gate and the one that just
failed, on the same shaft, with the counted bins outlined and the count that decided it.

### What "total" means

The electrodes the gate actually **saw**: the ungated table for the cohort's patients,
minus what `lf_concat` removes before the gate — non-neural channels, grid and
microelectrode contacts, excluded patients — and minus electrodes missing a condition.
Those filters are the pipeline's own, imported, and the result is checked: the total must
split exactly into the cohort and the gate's rejects, or the script stops.

**Reads** `outputs/_dataset/concat_source_v4/` (the ungated table and its params), the
`concat_hg` run's `labels.csv` for the cohort, the fsaverage coordinate table, and the
LanA atlas tables. **Writes** both PNGs, each with a `_caption.txt`; `_exemplars.csv`
beside the paper figure, `_patients.csv` and `_examples.csv` beside the supplement.


In [ ]:
# FIG 0, both figures. Reads the gate table, two electrodes' ERSP cubes for the gate
# pair and four for the exemplars; renders ten brains; nothing is refitted.
t0 = time.time()
sh(['00_paper2_figure0_coverage.py'])
print(f'\nFIG 0 (paper + supplement) in {time.time()-t0:.0f}s')

# one of the two:
# sh(['00_paper2_figure0_coverage.py', '--only', 'paper'])


---

# FIG 1 — the clustering, one figure per feature set

Four figures, **1a** to **1d**, one per feature set, **convex NMF only**, each cut at
**its own** held-out peak K. They are meant to be laid side by side, which is why panel
A is identical in all four.

| | feature set | features | K |
|---|---|---|---|
| **1a** | `concat_hg` — high gamma 70–150 Hz × time | 900 | 11 |
| **1b** | `concat_rawds` — 15 bands × time | 1350 | 12 |
| **1c** | `concat_bands5` — 5 bands × time | 450 | 14 |
| **1d** | `concat_bands5z` — 5 bands, z-scored per band | 450 | 13 |

### Panel A — held-out variance vs K

All four feature sets' convex-NMF curves, with this figure's emphasised and its ±1 SD
band shown. **Bi-cross-validated** (Owen & Perry 2009): rows *and* columns are held
out, so the score can fall as well as rise and the curve can turn over. A curve that
only rises cannot choose a K. The triangle on each curve is that curve's own peak; the
red dashed line is the K this figure is cut at.

Heights are comparable *within* a curve, not across them — each feature set is scored in
its own home space. What compares across curves is the **shape** and the **peak**.

### Panel B — one block per cluster

Each cluster gets a block, three rows deep: its **mean** on top, spanning the full block
width; underneath, the two hemisphere renders of the electrodes that carry it; and under
those, **who the cluster is made of** — one horizontal bar, one colour-coded segment per
patient, width proportional to that patient's electrodes in the cluster, widest first.
Response, anatomy and provenance are read together instead of in three rows a page apart.

**Block position means the same cluster in every figure.** Every run labels the identical
1693 electrodes in the identical order, so two solutions at one K can be matched 1:1 by
Hungarian assignment. `concat_bands5` (cNMF) defines the sequence — its own blocks
ordered by how alike a cluster's three condition profiles look, most alike first — and
every other figure is laid out so position *p* holds whatever its own cluster corresponds
to the reference's *p*-th. So block 1 of 1a, 1b, 1c and 1d is the same population of
electrodes, and the same will hold across algorithms once they're added.

Matching uses the **loading correlation** where both solutions are graded, falling back
to **shared electrodes** when either side is a hard partition (k-means, Ward), so it
still works when algorithms are added. Both are resolved the same way.

**Cluster numbers are unchanged**: c3 is still c3 in panel D, in the CSVs, in the run
report and on the 3-D brain; it just appears somewhere else on this page.

**The reference is taken at this figure's own K**, so both sides of a match have equal K
by construction. It follows that a set drawn at *different* K — the held-out peaks 11,
12, 14, 13 — is matched to a different reference solution in each figure, and **their
positions are not comparable with one another**. Only a set drawn at one shared K is,
which is what the K=8 set is for.

**A 1:1 assignment always returns a partner, whether or not one exists.** A cluster with
no real counterpart still gets paired, and position would then imply an agreement nothing
on the page contradicts — so the matched ids and per-cluster Jaccard overlaps go to the
caption and a `_matching.csv`, and *which clusters agree and which are left out* is
**Figure 2's** subject, not this one's.

`BLOCK_ORDER` at the top of the script takes `"matched"` (default), `"similarity"`
(each figure sorts its own) or `"cluster"` (numeric).

The **loading key** sits in the left margin, vertical, level with the first block's
renders — a legend for the column of brains beside it.

A cluster that is one patient's electrode strip is a single block of colour in that bar;
a cluster drawn from the cohort is a fine stripe. Patient colours come from `tab20` +
`tab20b` and are stable **within** a figure, so a patient is the same colour in every
cluster — but **not across** 1a–1d, because each figure sorts the patients present in its
own run.

- **The mean** is drawn in the representation the feature set was actually clustered in
  — a line in dB for `concat_hg`, frequency × time for the band sets. Three conditions
  concatenated, each time-normalised to 0–100 % of *its own* trial. **There is no
  absolute time axis**: trials differ in length and were warped to a common proportion,
  so the GO cue is at 50 % of every block and the axis is percent-of-trial, not seconds.
- **Dispersion across electrodes**, the same quantity in both representations: **±1
  standard deviation across the electrodes in the cluster**. `concat_hg` gets a shaded
  band; the band sets get an **outline enclosing the bins where `|mean| > SD`** — exactly
  the bins where that band would not span zero. The outline is drawn on **cell edges**,
  never between them: a contour would interpolate between band centres and imply a
  resolution a 5- or 15-band grid does not have. Condition blocks count as
  non-adjacent, so a region cannot appear to run from the end of one trial into the
  start of the next. SD, not SEM — with 50–436 electrodes a cluster the SEM only tells
  you the mean is well estimated, which was never in doubt.
- **`agree NN%` in each block title** — the share of bins inside that outline.
  **This is not a test and it is not evidence of generalization.** Electrodes on one
  patient's electrode strip are highly correlated, so a cluster that is mostly one
  person will tend to show *more* agreement, not less. Agreement asks whether the
  members look like their own centroid; **panel D** asks whether the members are more
  than one person. Read them together, never one instead of the other.
- **The renders** are pyvista/VTK on the fsaverage pial surface, with the material
  constants read live from `functions/lf_recon_shared_config.py` — the same ones the
  coverage and cluster renders use, so these cannot drift away from the visualizer.
  **The left hemisphere is seen from the left and the right from the right**, so each is
  a true lateral view: anterior falls left in the L render and right in the R render.
- **The fade.** An electrode's colour is the cluster colour mixed toward the grey every
  other electrode is drawn in, by
  `t = clip((w − 1/K) / (0.80 − 1/K), 0, 1)`, and its radius grows with `t`.
  At `t = 0` the loading is no better than a flat mixture over all K and the electrode
  is drawn *exactly* as a non-cluster one — it disappears by identity, not by
  transparency. The ramp saturating at **0.80** is a rendering choice: it changes how a
  loading is drawn, never the loading, the label, or any number reported.

### Panel C — how to read a cluster mean

**A trial strip on top**: what was on the screen, over the time it was there, read off
the paradigm figure (`Figure1_FBM_v1.png`). Each condition runs fixation → stimulus →
`?`, the `?` being the response cue. The stimulus box spans **0–50 %** of the block and
the `?` box **50–100 %**, so the boundary between them falls exactly on the dashed
GO-cue line below.

**The fixation screen is deliberately absent.** The paradigm has three screens per
trial; the strip shows two, because the warp is `proportions=(0.0, 0.5, 0.5)` and the
fixation phase is given **no time bins**. It hasn't been dropped for tidiness — it
occupies no part of the x axis. The baseline every dB value is expressed against
(−0.6 to −0.1 s) *does* come from it.

The audio, reading and `?` icons are drawn as vectors and stay sharp at any print size.
The picture stimulus is a real line drawing: drop a crop of it at
`outputs/clustering/paper_figures/assets/stim_picture.png` and it is used automatically;
without it a framed-image glyph stands in, so the figure builds either way.

**A schematic below.** Not any real cluster and not data — a low-frequency decrease and a
high-frequency increase locked to the GO cue, with amplitude growing across the three
conditions so the blocks are distinguishable. It names the remaining parts: the GO cue at
50 %, the percent-of-trial axis, and the frequency band edges (or the dB axis for
`concat_hg`).

### Panel D — does this K generalize?

**This panel is diagnostic, not decorative, and it chooses a K.** Computed at *every* K
the run was swept at, not only the published one. For each K the electrodes are assigned
by argmax of that K's loadings, and two numbers are taken:

- **red** — the share of **clusters** more than 50 % of whose electrodes come from one
  patient
- **orange** — the share of **electrodes** sitting inside those clusters

They answer different questions and can disagree: two of twenty small clusters failing is
not the same as two of five big ones, and only the second number tells them apart. The
grey curve on the right axis is panel A's held-out variance for this feature set, on the
same x.

**Why both curves have to be on one panel.** Held-out variance cannot see this failure.
Splitting a cohort until each patient has their own component fits held-out data
perfectly well — the components are real, they are just components of *individuals*
rather than of a population. So the held-out peak nominates a K, and this curve says what
that K costs. Where they disagree is a decision, not a computation.

The 50 % line is a **convention, not a test** — it is not calibrated against a null, it
is a threshold chosen to be readable. Clusters over it are titled in red in panel B.

### What it reads and what it writes

**Reads** — the newest `cnmf` run per feature set (`X_train.npy`,
`loadings_by_k/G_k<K>.npy`, `labels.csv`, `feature_schema.json`),
`heldout_variance_ALL.csv` and `heldout_peaks_cnmf.csv`, the fsaverage contact table and
the two pial surfaces.

**Writes**, into `outputs/clustering/paper_figures/`, for each of 1a–1d:

- `FIG1<x>_<fset>_cnmf_K<k>.png`
- `FIG1<x>_..._caption.txt` — full provenance and construction detail. Nothing is
  written on the figure itself, because a caption baked into a PNG cannot be edited in
  the manuscript or checked against the data.
- `FIG1<x>_..._patients.csv` — patient composition per cluster, at the published K.
- `FIG1<x>_..._generalization.csv` — panel D's curve: per K, how many clusters are one
  patient's and how many electrodes sit in them.
- `FIG1<x>_..._matching.csv` — which reference cluster each cluster was matched to, the
  Jaccard overlap of that pair, and the block position it was given. Written only when a
  match was possible.

Every write goes out through a verify-and-retry step: written locally, decode-checked,
copied, checked again. Writes to this share have truncated a finished file and reported
success more than once.

In [ ]:
# FIG 1a-1d. About 10 s per figure: ~25 offscreen renders plus 400 permutations per
# cluster for panel D's reference. Re-running is cheap and always safe - it reads the
# runs and writes only into paper_figures/.
t0 = time.time()
sh(['00_Paper2_Figures.py', '--figure', '1'])
print(f'\nall four figures in {time.time()-t0:.0f}s')

### Rebuilding one figure, or cutting at a different K

`--feature-set` takes any subset, `--k` overrides the held-out peak for every feature
set named. Useful for exactly the question panel D raises — whether the K that maximises
held-out variance is also the K that keeps clusters population-level.

In [ ]:
# one feature set only
# sh(['00_Paper2_Figures.py', '--figure', '1', '--feature-set', 'concat_hg'])

# the same feature set cut somewhere other than its held-out peak. The caption records
# that K was overridden, so a figure can never silently claim the peak it was not cut at.
# sh(['00_Paper2_Figures.py', '--figure', '1', '--feature-set', 'concat_bands5', '--k', '8'])

### Panel D, across all four figures

The `_patients.csv` and `_generalization.csv` files read back, so the diagnostic is in
the notebook and not only in the figures. This does not re-run anything.

`last_clean_K` is the largest K at which **no** cluster is more than half one patient —
the point past which the solution starts describing individuals. Compare it with the
held-out peak in the `K` column: where they disagree, that is the decision panel D puts
in front of you.

In [ ]:
import re
rows = []
for f in sorted(FIGDIR.glob('FIG1*_patients.csv')):
    pc = pd.read_csv(f)
    fs = f.stem.replace('_patients', '').split('_cnmf_')[0].split('_', 1)[1]
    K  = int(re.search(r'_K(\d+)_', f.name).group(1))
    worst = pc.loc[pc.top_share.idxmax()]
    r = dict(feature_set=fs, K=K,
             over_half=int((pc.top_share > 0.50).sum()),
             median_patients=int(pc.n_patients.median()),
             worst_cluster=f"c{int(worst.cluster)}",
             worst_share=round(float(worst.top_share), 3),
             worst_n_patients=int(worst.n_patients),
             random_draw_share=round(float(pc.null_share.mean()), 3))
    gf = f.with_name(f.name.replace('_patients.csv', '_generalization.csv'))
    if gf.exists():
        g = pd.read_csv(gf)
        at = g[g.k == K]
        r['pct_electrodes_at_K'] = (round(100 * float(at.frac_electrodes.iloc[0]))
                                    if len(at) else None)
        r['last_clean_K'] = (int(g[g.n_dominated == 0].k.max())
                             if (g.n_dominated == 0).any() else None)
    rows.append(r)

if rows:
    display(pd.DataFrame(rows).set_index('feature_set'))
    print('over_half           clusters where one patient supplies more than half the electrodes')
    print('pct_electrodes_at_K share of ALL electrodes sitting in those clusters, at this K')
    print('last_clean_K        largest K at which no cluster is more than half one patient')
    print('random_draw_share   what the largest patient supplies in a size-matched random draw')
else:
    print('no _patients.csv yet - run the FIG 1 cell above')

### What FIG 1 does not show

- **No statistical test of the clustering.** Separation against a matched null,
  anatomical coherence and leave-one-patient-out live in `249_cluster_statistics.ipynb`
  and are valid at **one K only**, because each is scored against a null refitted at
  that K. At the time of writing they exist for `concat_hg` and `concat_rawds` only —
  `concat_bands5` and `concat_bands5z` have **no statistics at any K**. That is the gap
  to close before these figures carry an argument.
- **Panel D is about patient composition**, which is necessary for a cluster to be a
  population response type and is not sufficient. A cluster can draw on every patient
  and still fail to replicate.
- **No comparison against k-means, Ward or archetypal analysis.** Panel A is convex NMF
  only.
- **The renders collapse depth.** An electrode deep in the temporal lobe and one on the
  lateral surface can overlap in the image.
- **The cluster label is an argmax of a graded loading.** The fade exists precisely
  because that argmax hides how weak most memberships are.

---

# FIG 2 — agreement: what survives, and what is left out

Built by **`00_paper2_figures2_2.py`**, which *imports* the matching machinery from
`00_Paper2_Figures.py` rather than reimplementing it — so FIG 1's block order and FIG 2's
correspondence can never disagree about which cluster is which.

**Two halves, the same three questions each.**

|  | varies | fixed |
|---|---|---|
| **top** | the four feature sets | convex NMF |
| **bottom** | the four algorithms | one feature set (`concat_hg` by default) |

`concat_hg` carries the algorithm half because **it is the only feature set with an
archetype run** — `concat_bands5` and `concat_bands5z` have none, so using FIG 1's
reference here would silently drop a whole algorithm.

### A / D — how much any two solutions agree at all

Adjusted Rand index between every pair. Chance-corrected: 0 is no better than random, 1
is identical. A single number for a whole partition — it says nothing about *which*
clusters agree, which is what B and E are for.

### B / E — which clusters survive, and which do not

Each solution's clusters are matched 1:1 to the reference's, and every pair's **Jaccard**
(shared electrodes over their union) is reported. Rows are the reference's clusters **in
FIG 1's block order**, so row *p* here is the cluster drawn at block position *p* there.

**A 1:1 assignment always returns a partner, whether or not one exists.** So a low value
is the interesting case — a cluster paired with something because it had to be, not
because the two describe the same electrodes. A row that is low across every column is a
cluster only the reference found. That is the "what is left out" this panel exists for.

### C / F — which electrodes are placed consistently

Every solution's labels are translated into the reference's numbering, then each
electrode is scored by the size of the largest group of solutions that agree on it.
1 = no two agree, N = all of them do. Red to green, shown on both hemispheres with the
distribution beside them.

It is the **modal** assignment, not "agrees with the reference" — nothing privileges one
solution, so an electrode where the reference is the odd one out still reads as agreement
among the rest. And it is **not the same question as B/E**: a cluster can match well on
average and still be assembled from electrodes no other solution groups together.

### What it writes

`FIG2_agreement_K<k>.png`, plus `_caption.txt`, `_cluster_agreement.csv` (every
cluster × solution Jaccard) and `_electrode_agreement.csv` (per-electrode counts for both
halves).

**Requires equal K across everything compared** — 1:1 matching has no meaning otherwise —
so the figure is built at one K at a time. Solutions that are missing, at a different K,
or over a different electrode set are dropped with a message rather than compared.

In [ ]:
# FIG 2 at K=8, one figure per feature set carrying the ALGORITHM half. Each reads
# the four cNMF runs and that feature set's four algorithm runs and renders six brains;
# nothing is refitted. The four are meant to be compared, and every matrix on them
# runs 0-1 for that reason.
t0 = time.time()
for fset in FEATURE_SETS:
    sh(['00_paper2_figures2_2.py', '--k', '8', '--algo-feature-set', fset])
print(f'\nFIG 2, four variants, in {time.time()-t0:.0f}s')

# a single variant, or another K:
# sh(['00_paper2_figures2_2.py', '--k', '8', '--algo-feature-set', 'concat_hg'])
# sh(['00_paper2_figures2_2.py', '--k', '12'])


---

# FIG 3 — the LanA language atlas

Built by **`00_paper2_figure3_lana.py`**, one figure per feature set. How far into the
probabilistic language atlas each cluster sits, and whether belonging more strongly to a
cluster goes with sitting further in. Four panels, one per corner.

### A — clusters ranked by how much LanA they sit in

Mean P(LanA) of each cluster's electrodes, most to least, whiskers from bootstrapping
**patients**. Two nulls: a **shaft-shift** that rolls each shaft's labels along itself and
keeps the spatial smoothness that makes neighbouring contacts alike (the one that counts,
`*`), and a within-patient shuffle that does not (`(*)`).

### B — those correlations, ranked

Spearman ρ between loading and P(LanA), effect size first: the shaded band is |ρ| < 0.10.
The K values are **not independent** — loadings sum to 1, so one cluster tracking the atlas
forces the others negative by arithmetic.

### C — loading vs P(LanA), every electrode

One scatter per cluster, two rows.

### D — LanA on this coverage

Every electrode with a LanA value, coloured by it: left, right, and **from above**.

### One set of axes for all four

A's y, B's x and C's x and y run to the **same limits in every FIG 3**, so the four can
be laid side by side. `--all` analyses all four feature sets in one process, takes the
global extents, renders all four on them, and records the extents in
`FIG3_extents_K08.json`; a later single-feature-set run reads that file and stays on the
shared axes. **Rebuild all four together** whenever any of them changes.

**Reads** the four cNMF runs at this K and the LanA atlas tables under
`outputs/clustering/atlas/`. **Writes** `FIG3_lana_<fset>_K8.png`, `_clusters.csv`,
`_electrodes.csv`, `_caption.txt`, and the shared `FIG3_extents_K08.json`.


In [ ]:
# FIG 3, all four feature sets on one set of axes. Two nulls x 1000 permutations and
# 1000 patient bootstraps per feature set, so a few minutes in all; three brains each.
t0 = time.time()
sh(['00_paper2_figure3_lana.py', '--all', '--k', '8'])
print(f'\nFIG 3, four feature sets, in {time.time()-t0:.0f}s')

# one feature set, on the axes recorded by the last --all:
# sh(['00_paper2_figure3_lana.py', '--feature-set', 'concat_hg', '--k', '8'])


---

# Adding the next figure

Figure 1 lives in `00_Paper2_Figures.py` (registered in its `FIGURES` dict, so
`--figure 1` selects it). Figure 2 lives in its **own file** and imports what it needs
from Figure 1's. Either pattern is fine; what matters is:

1. **The drawing goes in a `.py`, never in a notebook cell.** A figure that can only be
   rebuilt by running a cell cannot be rebuilt on the server.
2. **Import shared machinery, don't copy it.** `match_clusters`, `_scene`,
   `condition_similarity`, the palette and the verified writers are all importable. Two
   copies of the matching code is two answers to "which cluster is this".
3. **Nothing is written on the figure.** Every figure gets a sibling `_caption.txt`, and
   the numbers behind each panel get a CSV.
4. **Route every output through `save_png` / `save_text`** so it is verified on the way
   out — this share has truncated finished files and reported success.
5. **Add a section here**: what the figure is, panel by panel; what it reads; what it
   writes; then one cell that calls the script.